In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
CATALOG = 'workspace'
SCHEMA = 'unity_airways'

# Chapter 6: Evaluating GenAI applications within MLflow

GenAI applications represent a paradigm shift from traditional ML models. Unlike classic models that produce structured, deterministic outputs, GenAI systems generate natural language responses that must be evaluated for qualities like relevance, helpfulness, factual accuracy, and user satisfaction.

### Why Traditional Metrics Miss GenAI Application Behaviour

Traditional ML evaluation metrics (accuracy, precision, recall, F1-score) were designed for classification and regression tasks with clear ground truth labels. These metrics fall short for GenAI applications because:

1. **Deterministic vs. Generative Outputs**: Traditional models produce fixed outputs for given inputs, while GenAI models generate variable, creative responses
2. **Structured vs. Unstructured Data**: Traditional metrics work with numerical or categorical outputs, not natural language text
3. **Single Correct Answer vs. Multiple Valid Responses**: GenAI tasks often have many acceptable answers, making binary accuracy insufficient
4. **Context and Nuance**: Traditional metrics don't capture semantic meaning, tone, helpfulness, or user experience
5. **Safety and Ethics**: GenAI outputs must be evaluated for harmful content, bias, and policy compliance

### What are the Components to Evaluate

GenAI applications typically consist of multiple components that require different evaluation approaches:

1. **Retrieval Components**: 
   - Retrieval accuracy and relevance
   - Document ranking quality
   - Coverage of relevant information

2. **Generation Components**:
   - Factual correctness and groundedness
   - Relevance to user query
   - Coherence and fluency
   - Tone and style appropriateness

3. **End-to-End System**:
   - User experience and satisfaction
   - Task completion effectiveness
   - Safety and policy compliance
   - Latency and performance

4. **Business Logic**:
   - Adherence to guidelines and policies
   - Consistency across similar queries
   - Integration with downstream systems

### Evaluation Modes: Direct Evaluation vs Answer Sheet Evaluation

**Direct Evaluation:**
- Assesses model outputs directly against criteria or guidelines
- Uses LLM judges to evaluate qualities like helpfulness, relevance, safety
- Suitable for open-ended tasks without single correct answers
- Examples: Chatbot responses, creative writing, summarization

**Answer Sheet Evaluation:**
- Compares model outputs to curated reference answers
- Uses exact match, semantic similarity, or custom comparison functions
- Suitable for tasks with clear correct answers
- Examples: Question answering, fact extraction, classification

## Use Case: Unity Airways Customer Service Agent

This notebook demonstrates best practices for evaluating a GenAI-powered customer service agent for Unity Airways (a fictional airline). The agent helps customers with:
- Flight bookings and modifications
- Policy questions (baggage, refunds, cancellations)
- General customer support inquiries

We'll evaluate using both structured booking data and unstructured FAQ/QA datasets, demonstrating the full spectrum of GenAI evaluation techniques in MLflow 3+.

## Creating & Managing Evaluation Datasets

Evaluation datasets are the foundation of GenAI application testing. MLflow 3+ provides powerful tools for creating, managing, and versioning evaluation datasets that enable systematic testing and continuous improvement.

### Dataset Types and Sources

**1. Production Trace Datasets**
- Built from real user interactions captured by MLflow Tracing
- Provides authentic user scenarios and edge cases
- Enables testing against actual production patterns

**2. Curated Test Datasets**
- Manually created examples targeting specific features or edge cases
- Ground truth answers for answer sheet evaluation
- Domain expert validated responses

**3. Synthetic Datasets**
- Generated using LLMs to expand test coverage
- Useful for testing rare scenarios or adversarial cases
- Can simulate different user personas and interaction styles

### Approaches to Building Evaluation Datasets

MLflow 3+ offers several flexible approaches to construct evaluation datasets:

#### Approach 1: Build from Existing Traces

One of the most effective ways to build relevant evaluation datasets is by curating examples from your application's historical interactions captured by MLflow Tracing.

```python
import mlflow
import time

# Search for traces from the last hour
one_hour_ago = int((time.time() - 60 * 60) * 1000)

traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {one_hour_ago} AND "
                 f"attributes.status = 'OK'",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=100
)

# Add traces to evaluation dataset
eval_dataset.merge_records(traces)
```

#### Approach 2: Build from Scratch or Import Existing

You can import existing datasets or create examples from scratch. Data must match the evaluation dataset schema:

```python
evaluation_examples = [
    {
        "inputs": {"question": "What is the baggage allowance for international flights?"},
        "expected": {
            "expected_response": "For international flights, you can bring one carry-on bag (22x14x9 inches) and one personal item. Checked baggage allowance varies by fare type.",
            "expected_categories": ["baggage", "international", "policy"]
        }
    },
    {
        "inputs": {"question": "How do I cancel my flight?"},
        "expected": {
            "expected_response": "You can cancel your flight online through Manage My Booking, by calling customer service, or at the airport. Cancellation fees may apply.",
            "expected_categories": ["cancellation", "booking", "policy"]
        }
    }
]

eval_dataset.merge_records(evaluation_examples)
```

#### Approach 3: Synthesize Evaluation Sets

Generate synthetic data to expand testing coverage and create diverse scenarios:

```python
from mlflow.genai.datasets import synthesize_dataset

# Generate synthetic customer service scenarios
synthetic_dataset = synthesize_dataset(
    base_examples=evaluation_examples,
    num_examples=50,
    persona_variations=["frustrated customer", "first-time flyer", "business traveler"],
    scenario_types=["booking", "cancellation", "policy inquiry", "complaint"]
)

eval_dataset.merge_records(synthetic_dataset)
```

#### Approach 4: Domain Expert Labels

Leverage feedback from domain experts captured in MLflow Labeling Sessions:

```python
import mlflow.genai.labeling as labeling

# Get labeling sessions
labeling_sessions = labeling.get_labeling_sessions()

# Sync labeled data to evaluation dataset
for session in labeling_sessions:
    if session.name == "Unity Airways Customer Service Review":
        session.sync(dataset_name="catalog.schema.unity_airways_eval")
```

### Creating MLflow Evaluation Datasets

Let's demonstrate creating a managed evaluation dataset from our Unity Airways data:

```python
import mlflow.genai.datasets

# Create evaluation dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name=f"{CATALOG}.{SCHEMA}.unity_airways_evaluation_dataset",
    name="Unity Airways Customer Service Evaluation",
    description="Comprehensive evaluation dataset for Unity Airways customer service agent"
)
```

## Practical Implementation: Unity Airways Customer Service Agent

Let's now implement a complete GenAI evaluation workflow using our Unity Airways customer service agent. This section demonstrates all the concepts covered in this chapter with working code.

### Step 1: Create a Customer Service Agent

First, let's create a simple customer service agent that can answer common questions about Unity Airways policies and services.


In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

# Enable MLflow's autologging to instrument your application with Tracing
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/chapter-6")

# Create an OpenAI client that is connected to Databricks-hosted LLMs
w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()

# Select an LLM
model_name = "databricks-gpt-oss-120b"

In [0]:
# Unity Airways Customer Service Agent Implementation
import mlflow.genai.datasets
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines
import pandas as pd
import time
from typing import Dict, List

# Enable MLflow tracing
mlflow.openai.autolog()

# Unity Airways Knowledge Base 
UNITY_AIRWAYS_KB = {
    "baggage": {
        "carry_on": "One carry-on bag (22x14x9 inches) and one personal item allowed",
        "checked": "First checked bag free for premium customers, $30 for economy",
        "weight_limit": "50 lbs for checked bags, no weight limit for carry-on"
    },
    "cancellation": {
        "policy": "Cancel within 24 hours for full refund. After 24h, fees apply based on fare type",
        "fees": "Economy: $200, Premium: $100, First Class: No fee",
        "process": "Cancel online, by phone 1-800-UNITY-AIR, or at airport"
    },
    "booking": {
        "online": "Book at unityairways.com or mobile app",
        "phone": "Call 1-800-UNITY-AIR for assistance",
        "changes": "Changes allowed up to 2 hours before departure"
    }
}

@mlflow.trace
def unity_airways_customer_service_agent(customer_question: str) -> Dict[str, str]:
    """
    Unity Airways customer service agent that answers common questions.
    This is a simplified rule-based agent for demonstration purposes.
    """
    question_lower = customer_question.lower()
    
    # Simple keyword-based routing
    if any(word in question_lower for word in ["baggage", "bag", "luggage"]):
        if "carry" in question_lower or "cabin" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["carry_on"]
        elif "checked" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["checked"]
        elif "weight" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["weight_limit"]
        else:
            response = "For baggage information: " + UNITY_AIRWAYS_KB["baggage"]["carry_on"]
    
    elif any(word in question_lower for word in ["cancel", "cancellation", "refund"]):
        if "fee" in question_lower or "cost" in question_lower:
            response = UNITY_AIRWAYS_KB["cancellation"]["fees"]
        elif "how" in question_lower or "process" in question_lower:
            response = UNITY_AIRWAYS_KB["cancellation"]["process"]
        else:
            response = UNITY_AIRWAYS_KB["cancellation"]["policy"]
    
    elif any(word in question_lower for word in ["book", "booking", "reservation"]):
        if "change" in question_lower or "modify" in question_lower:
            response = UNITY_AIRWAYS_KB["booking"]["changes"]
        elif "phone" in question_lower:
            response = UNITY_AIRWAYS_KB["booking"]["phone"]
        else:
            response = UNITY_AIRWAYS_KB["booking"]["online"]
    
    else:
        response = "Thank you for contacting Unity Airways. For immediate assistance, please call 1-800-UNITY-AIR or visit our website at unityairways.com"
    
    # Add helpful closing
    response += " Is there anything else I can help you with today?"
    
    return {"response": response}

# Test the agent
test_question = "What's the baggage policy for carry-on items?"
result = unity_airways_customer_service_agent(test_question)
print(f"Question: {test_question}")
print(f"Response: {result['response']}")


### Step 2: Create Comprehensive Evaluation Dataset

Now let's create a comprehensive evaluation dataset that covers different types of customer service scenarios.

In [0]:
# Read evaluation data from Unity Airways QA dataset in Unity Catalog
eval_data_df = spark.read.table(
    f"{CATALOG}.{SCHEMA}.qa_dataset"
)

# Transform columns to match agent evaluation schema
eval_data = eval_data_df.selectExpr(
    "struct(Question as customer_question) as inputs",
    "struct(Answer as expected_response) as expected"
)

# Sample 20 examples for evaluation and convert to Pandas DataFrame
eval_df = eval_data.limit(20).toPandas()

# Display the sampled evaluation dataset
display(eval_df)

### Step 3: Define Comprehensive Scorers

Let's create a mix of LLM-based and code-based scorers to evaluate our customer service agent comprehensively.


In [0]:
# Define comprehensive scorers for Unity Airways customer service evaluation

from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines, scorer
from mlflow.entities import Feedback

# Custom Guidelines-Based Scorers
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="""
    The customer service response must use professional, courteous language appropriate for airline customer service.
    Requirements:
    - Use polite and respectful language
    - Avoid casual expressions or slang
    - Maintain helpful and solution-oriented tone
    - Include appropriate greetings/closings when relevant
    """
)

brand_compliance_scorer = Guidelines(
    name="brand_compliance",
    guidelines="""
    The response must follow Unity Airways brand guidelines:
    - Mention Unity Airways when appropriate
    - Use consistent contact information (1-800-UNITY-AIR, unityairways.com)
    - Maintain positive, customer-focused messaging
    - Provide specific, actionable information when possible
    """
)

completeness_scorer = Guidelines(
    name="response_completeness", 
    guidelines="""
    The customer service response must completely address the customer's question:
    - Directly answer the specific question asked
    - Provide all relevant details mentioned in the expected response
    - Include next steps or additional resources when appropriate
    - Avoid generic responses when specific information is requested
    """
)

# 2. Code-Based Scorers
@scorer
def response_length_checker(outputs) -> Feedback:
    """Check if response length is appropriate (not too short or too long)."""
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    word_count = len(response.split())
    
    if word_count < 5:
        return Feedback(value = 0, rationale = f"Response too short ({word_count} words)")
    elif word_count > 100:
        return Feedback(value =  0.5, rationale = f"Response quite long ({word_count} words)")
    else:
        return Feedback(value =  1, rationale = f"Appropriate length ({word_count} words)")

@scorer
def contact_info_checker(outputs) -> Feedback:
    """Check if response includes appropriate Unity Airways contact information."""
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    response_lower = response.lower()
    
    has_phone = "1-800-unity-air" in response_lower
    has_website = "unityairways.com" in response_lower
    mentions_contact = any(word in response_lower for word in ["call", "phone", "website", "visit"])
    
    if has_phone and has_website:
        return Feedback(value = 1, rationale = "Includes both phone and website")
    elif has_phone or has_website:
        return Feedback(value = 0.8, rationale = "Includes one form of contact info")
    elif mentions_contact:
        return Feedback(value = 0.3, rationale = "Mentions contacting but no specific info")
    else:
        return Feedback(value = 0, rationale = "No contact information provided")

@scorer
def policy_accuracy_checker(outputs, expectations=None) -> Feedback:
    """Check if the response contains accurate policy information."""
    if not expectations:
        return Feedback(value = 0.5, rationale = "No expected response to compare against")
    
    response = outputs if isinstance(outputs, str) else outputs.get('response', '')
    expected_response = expectations.get('expected_response', '') if isinstance(expectations, dict) else str(expectations)
    
    # Simple keyword overlap check
    response_words = set(response.lower().split())
    expected_words = set(expected_response.lower().split())
    
    # Remove common words
    common_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are'}
    response_words -= common_words
    expected_words -= common_words
    
    if not expected_words:
        return Feedback(value = 0.5, rationale = "No meaningful expected words to compare")
    
    overlap = len(response_words & expected_words) / len(expected_words)
    
    if overlap >= 0.7:
        return Feedback(value = 1, rationale = f"High accuracy ({overlap:.2f} keyword overlap)")
    elif overlap >= 0.4:
        return Feedback(value = 0.7, rationale = f"Good accuracy ({overlap:.2f} keyword overlap)")
    elif overlap >= 0.2:
        return Feedback(value = 0.4, rationale = f"Partial accuracy ({overlap:.2f} keyword overlap)")
    else:
        return Feedback(value = 0, rationale = f"Low accuracy ({overlap:.2f} keyword overlap)")

# Combine all scorers
unity_airways_scorers = [
    # LLM-based scorers
    RelevanceToQuery(),
    Safety(),
    professional_tone_scorer,
    brand_compliance_scorer,
    completeness_scorer,
    
    # Code-based scorers  
    response_length_checker,
    contact_info_checker,
    policy_accuracy_checker
]

print(f"Created {len(unity_airways_scorers)} scorers for Unity Airways evaluation:")
for scorer in unity_airways_scorers:
    print(f"- {scorer.name}: {'LLM-based' if hasattr(scorer, 'guidelines') or scorer.name in ['relevance_to_query', 'safety'] else 'Code-based'}")

# Test a scorer
test_response = "One carry-on bag (22x14x9 inches) and one personal item allowed. Is there anything else I can help you with today?"
test_expected = {"expected_response": "One carry-on bag (22x14x9 inches) and one personal item allowed"}

accuracy_result = policy_accuracy_checker(test_response, test_expected)
print(f"\nTest scorer result: {accuracy_result}")


### Step 4: Run Complete Evaluation and Analysis

Now let's run a complete evaluation of our Unity Airways customer service agent and analyze the results comprehensively.


In [0]:
eval_results = mlflow.genai.evaluate(
    data=eval_df,
    predict_fn=unity_airways_customer_service_agent,
    scorers=unity_airways_scorers
)